# Criação da tabela Gold - TIMES

## Objetivos

- Agrupamento de estatísticas gerais por time
- Classificação do campeonato, considerando alguns critérios de desempate (saldo de gols, gols marcados, cartões amarelos e cartões vermelhos)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df_cards_team = (
    spark.read.table("api_football.silver.cards")
    # agrupamento devido agregação (soma)
    .groupBy("team_id")
    .agg(
        # soma de cartões amarelos por time: quando a linha da coluna card_type está como "yellow card" soma um, caso contrário soma zero
        sum(when(col("card_type") == "yellow card", 1).otherwise(0)).alias( "yellow_cards"),
        # soma de cartões vermelhos por time (mesmo processo anterior)
        sum(when(col("card_type") == "red card", 1).otherwise(0)).alias("red_cards"),
    )
)

# display(df_cards_team)

In [0]:
df_stats_team = (
    spark.read.table("api_football.silver.statistics")
    # agrupamento devido agregação
    .groupBy("team_id")
    .agg(
        # soma de escanteios por time
        sum(col("corners")).alias("corners"),
        # soma de penaltis por time
        sum(col("penalty")).alias("penalty"),
        # soma de substituições por time
        sum(col("substitutions")).alias("substitutions"),
        # soma de impedimentos por time com tratamento de valores nulos
        coalesce(sum(col("offsides")), lit(0)).alias("offsides"),
    )
)

# display(df_stats_team)

In [0]:
df_matches = spark.read.table("api_football.silver.matches")

In [0]:
df_home = df_matches.select(
    "match_id",
    col("match_hometeam_id").alias("team_id"),
    col("match_hometeam_name").alias("team_name"),
    # transformação de datatype pra inteiro
    col("match_hometeam_score").cast("int").alias("goals_for"),
    # transformação de datatype pra inteiro
    col("match_awayteam_score").cast("int").alias("goals_against"),
    # se a coluna match_result for "home, draw ou loss" soma um, caso contrário soma zero e salva na coluna correspondente
    when(col("match_result") == "home", 1).otherwise(0).alias("win"),
    when(col("match_result") == "draw", 1).otherwise(0).alias("draw"),
    when(col("match_result") == "away", 1).otherwise(0).alias("loss"),
    # quando a coluna match_result for "home" adiciona o valor 3 
    when(col("match_result") == "home", 3)
    # quando a coluna match_result for "draw" adiciona o valor 1 
    .when(col("match_result") == "draw", 1)
    # caso contrário adiciona o valor 0
    .otherwise(0)
    # salva na coluna points
    .alias("points"),
).orderBy("match_id")

# display(df_home)

In [0]:
# mesmo processo do anterior, porém verifica o campo "away", ao invés do "home"

df_away = (df_matches.select(
        "match_id",
        col("match_awayteam_id").alias("team_id"),
        col("match_awayteam_name").alias("team_name"),
        col("match_awayteam_score").cast("int").alias("goals_for"),
        col("match_hometeam_score").cast("int").alias("goals_against"),
        when(col("match_result") == "away", 1).otherwise(0).alias("win"),
        when(col("match_result") == "draw", 1).otherwise(0).alias("draw"),
        when(col("match_result") == "home", 1).otherwise(0).alias("loss"),
        when(col("match_result") == "away", 3)
        .when(col("match_result") == "draw", 1)
        .otherwise(0)
        .alias("points"),
    )
).orderBy("match_id")

# display(df_away)

foi necessária a separação em dois dataframes para possibilitar o uso da chave team_id para realização dos joins para visualização dos dados por time, uma vez que a tabela matches tem esse dado misturado em duas colunas (match_awayteam_id e match_hometeam_id)

In [0]:
df_team_matches = (df_home.unionByName(df_away)).orderBy("match_id")

#display(df_team_matches)

In [0]:
df_team_ranking = (
    # agrupamento dos dados por time
    df_team_matches.groupBy("team_id", "team_name")
    .agg(
        count("*").alias("matches"),
        sum("win").alias("wins"),
        sum("draw").alias("draws"),
        sum("loss").alias("losses"),
        sum("points").alias("points"),
        sum("goals_for").alias("goals_for"),
        sum("goals_against").alias("goals_against"),
    )
    # adiciona uma coluna com o saldo de gols
    .withColumn("goal_difference", (col("goals_for") - col("goals_against")))
    # ordena por pontos e saldo de gols
    .orderBy(col("points").desc(), col("goal_difference").desc())
)

# display(df_team_ranking)

In [0]:
# cria uma janela ordenada pelos critérios de desempate (pontos, saldo de gols, gols marcados, cartões amarelos e cartões vermelhos)
window_spec = Window.orderBy(
    col("points").desc(),
    col("goal_difference").desc(),
    col("goals_for").desc(),
    col("red_cards").asc(),
    col("yellow_cards").asc(),
)

df_gold_team = (
    df_team_ranking.alias("r")
    #todos os times estão presentes nos três dataframes, então usa left join
    .join(df_cards_team.alias("c"), "team_id", "left")
    .join(df_stats_team.alias("s"), "team_id", "left")
    # cria uma coluna com o ranking do campeonato usando os dados da janela
    .withColumn("ranking", rank().over(window_spec))
    # seleção de colunas do dataframe
    .select(
        "ranking",
        "r.*",
        "c.yellow_cards",
        "c.red_cards",
        "s.corners",
        "s.penalty",
        "s.substitutions",
        "s.offsides",
    )
)

# display(df_gold_team)

In [0]:
df_gold_team.write.mode("overwrite").saveAsTable("api_football.gold.team_statistics")

In [0]:
%sql
-- select
--   *
-- from
--   api_football.gold.team_statistics